# CropSure Research Notebook 05: Random Forest Agronomic Yield-Loss Prediction

## Context & Agronomic Rationale
In precision agriculture, automated disease detection is merely the first diagnostic step. The primary commercial imperative for farmers, agricultural cooperatives, and crop insurance underwriters is quantifying the projected **economic and yield loss** ($0.0 - 100.0\%$) resulting from a disease outbreak.

Crop loss is not a linear function of instantaneous leaf spot count; it is a multivariate consequence of:
1. **Cumulative Disease Pressure (AUDPC)**: Sustained historical infection that exhausts photosynthetic leaf area.
2. **Phenological Growth Stage**: Infection during flowering (Stage 3) or fruit development (Stage 4) induces flower abortion and catastrophic yield loss compared to late-season infection during harvest ripening (Stage 6).
3. **Remediation Latency**: Days elapsed between initial pathogen sporulation and active chemical/biological intervention.
4. **Vapor Pressure Deficit (VPD)**: Atmospheric transpirational demand that compounds foliar water stress.

## Research Objectives
1. Synthesize a calibrated agronomic corpus of 2,000 crop observation records.
2. Execute a stratified 80/20 train/test split preserving disease severity distribution across partitions.
3. Train a tuned `RandomForestRegressor` ensemble with 5-fold cross-validation.
4. Perform feature importance ranking (Mean Decrease in Impurity & Permutation Importance).
5. Export production joblib artifacts and demonstrate real-world advisory loss forecasting.

In [ ]:
# Cell 02: Core Environment Imports
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
print(f"[IMPORT] Scikit-Learn: {sys.modules['sklearn'].__version__} | Joblib: {joblib.__version__}")

In [ ]:
# Cell 03: Seed Fixation and Directory Path Setup
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

NOTEBOOK_DIR = Path(os.path.abspath("")).resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"[INIT] Root Project Directory : {PROJECT_ROOT}")
print(f"[INIT] Models Directory       : {MODELS_DIR}")

## Feature Engineering & Mechanistic Loss Formulation

The regressor maps 7 primary physiological and environmental indicators to harvest yield loss percentage:

| Feature Symbol | Description | Agronomic Range | Measurement Unit |
| :--- | :--- | :--- | :--- |
| `audpc` | Area Under Disease Progress Curve | $0.0 - 100.0$ | $\% \cdot \text{days}$ |
| `phenology_stage` | Growth Stage ($1$=Seedling, $2$=Vegetative, $3$=Flowering, $4$=Fruit Development, $5$=Ripening, $6$=Harvest) | $1 - 6$ | Categorical / Ordinal |
| `remediation_latency_days` | Elapsed days prior to chemical remediation | $0 - 30$ | Days |
| `vpd` | Vapor Pressure Deficit | $0.5 - 4.0$ | $\text{kPa}$ |
| `disease_severity` | Instantaneous foliar necrotic fraction | $0.0 - 1.0$ | Unitless ratio |
| `temp_avg` | Seasonal mean canopy temperature | $15.0 - 35.0$ | $^\circ\text{C}$ |
| `humidity_avg` | Seasonal mean relative humidity | $40.0 - 100.0$ | $\%$ |

### Ground Truth Yield Loss Equation
Synthetic ground truth target values are synthesized according to the physiological response function:

$$\text{YieldLoss}(\%) = 0.40 \cdot \text{audpc} + 0.20 \cdot (\text{severity} \cdot 100) + 0.15 \cdot (\text{latency} \cdot 2.5) + 0.10 \cdot \max(0, (\text{vpd} - 1.5) \cdot 10) + \text{stage\_penalty} + \epsilon$$

where $\epsilon \sim \mathcal{N}(0, 2.5)$ represents stochastic field micro-heterogeneity, bounded strictly within $[0.0, 100.0]\%$.

In [ ]:
# Cell 05: Synthesize 2,000 Calibrated Agronomic Observations
NUM_SAMPLES = 2000
np.random.seed(RANDOM_SEED)

print(f"[SYNTHESIS] Generating {NUM_SAMPLES} calibrated agronomic yield loss records...")

# 1. Instantaneous severity ~ Beta distribution (heavy left skew towards mild infections)
disease_severity = np.random.beta(a=1.5, b=3.5, size=NUM_SAMPLES)

# 2. AUDPC strongly correlates with severity with non-linear variance
audpc = np.clip(disease_severity * 85.0 + np.random.normal(0, 8.0, size=NUM_SAMPLES), 0.0, 100.0)

# 3. Phenology stage 1 to 6
phenology_stage = np.random.randint(1, 7, size=NUM_SAMPLES)

# 4. Remediation latency (days)
remediation_latency = np.random.exponential(scale=6.0, size=NUM_SAMPLES)
remediation_latency = np.clip(remediation_latency, 0.0, 30.0)

# 5. Microclimate (VPD, Temperature, Humidity)
temp_avg = np.random.uniform(15.0, 35.0, size=NUM_SAMPLES)
humidity_avg = np.random.uniform(40.0, 100.0, size=NUM_SAMPLES)

# Approximate saturated vapor pressure Tetens equation: es = 0.61078 * exp(17.27 * T / (T + 237.3))
es = 0.61078 * np.exp((17.27 * temp_avg) / (temp_avg + 237.3))
ea = es * (humidity_avg / 100.0)
vpd = np.clip(es - ea, 0.5, 4.0)

# 6. Phenological stage vulnerability penalty (Stages 3 and 4 have highest vulnerability)
stage_penalties = {1: 2.0, 2: 5.0, 3: 15.0, 4: 18.0, 5: 8.0, 6: 3.0}
stage_vuln = np.array([stage_penalties[s] for s in phenology_stage])

# 7. Mechanistic target yield loss calculation
noise = np.random.normal(0.0, 2.5, size=NUM_SAMPLES)
yield_loss_raw = (
    0.40 * audpc +
    0.20 * (disease_severity * 100.0) +
    0.15 * (remediation_latency * 2.5) +
    0.10 * np.maximum(0.0, (vpd - 1.5) * 10.0) +
    stage_vuln +
    noise
)
yield_loss_pct = np.clip(yield_loss_raw, 0.0, 100.0)

FEATURE_COLUMNS = [
    "audpc",
    "phenology_stage",
    "remediation_latency_days",
    "vpd",
    "disease_severity",
    "temp_avg",
    "humidity_avg"
]

df_yield = pd.DataFrame({
    "audpc": np.round(audpc, 2),
    "phenology_stage": phenology_stage,
    "remediation_latency_days": np.round(remediation_latency, 1),
    "vpd": np.round(vpd, 3),
    "disease_severity": np.round(disease_severity, 4),
    "temp_avg": np.round(temp_avg, 2),
    "humidity_avg": np.round(humidity_avg, 2),
    "yield_loss_pct": np.round(yield_loss_pct, 2)
})

print(f"[SYNTHESIS] Successfully constructed {len(df_yield)} observation records.")
print(df_yield.describe().T[["mean", "std", "min", "50%", "max"]])

In [ ]:
# Cell 06: Stratified Train / Test Partitioning (80/20 Split)
# Stratify by 5 disease severity quantiles to preserve severity spectrum across train/test splits
severity_bins = pd.qcut(df_yield["disease_severity"], q=5, labels=False)

X = df_yield[FEATURE_COLUMNS]
y = df_yield["yield_loss_pct"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=severity_bins
)

print(f"[SPLIT] Data Partitioning Completed:")
print(f"  - Training Corpus   : {X_train.shape[0]} samples (80%)")
print(f"  - Evaluation Corpus : {X_test.shape[0]} samples (20%)")
print(f"  - Feature Count     : {X_train.shape[1]}")

In [ ]:
# Cell 07: Train RandomForestRegressor with 5-Fold Cross-Validation
rf_hyperparameters = {
    "n_estimators": 200,
    "max_depth": 15,
    "min_samples_leaf": 5,
    "random_state": RANDOM_SEED,
    "n_jobs": -1
}

print(f"[TRAIN] Initializing RandomForestRegressor with parameters: {rf_hyperparameters}...")
model_rf = RandomForestRegressor(**rf_hyperparameters)

# 5-Fold Cross-Validation on Training Partition
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_r2_scores = cross_val_score(model_rf, X_train, y_train, cv=kfold, scoring="r2")
cv_mae_scores = -cross_val_score(model_rf, X_train, y_train, cv=kfold, scoring="neg_mean_absolute_error")

print(f"[CV] 5-Fold Cross-Validation Results:")
print(f"  - Mean R² Score : {cv_r2_scores.mean():.4f} (± {cv_r2_scores.std():.4f})")
print(f"  - Mean MAE      : {cv_mae_scores.mean():.4f}% (± {cv_mae_scores.std():.4f}%)")

# Fit Final Model on Complete Training Set
start_fit = time.time()
model_rf.fit(X_train, y_train)
fit_time = time.time() - start_fit
print(f"[TRAIN] Final model fitting completed in {fit_time:.2f} seconds.")

In [ ]:
# Cell 08: Test Set Evaluation & Residual Diagnostics
y_pred = model_rf.predict(X_test)

test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2 = r2_score(y_test, y_pred)
test_mape = np.mean(np.abs((y_test - y_pred) / np.maximum(y_test, 1.0))) * 100.0

print("\n==================================================")
print("       Random Forest Yield-Loss Test Metrics       ")
print("==================================================")
print(f"  - Mean Absolute Error (MAE)   : {test_mae:.3f} %")
print(f"  - Root Mean Squared Error     : {test_rmse:.3f} %")
print(f"  - Coefficient of Det. (R²)    : {test_r2:.4f}")
print(f"  - Mean Abs. Pct Error (MAPE)  : {test_mape:.2f} %")
print("==================================================")

# Diagnostic Visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Predicted vs Actual Scatter
axes[0].scatter(y_test, y_pred, alpha=0.55, color="darkolivegreen", edgecolors="none", s=40)
axes[0].plot([0, 100], [0, 100], color="crimson", linestyle="--", lw=2.5, label="1:1 Perfect Fidelity")
axes[0].set_title(f"Predicted vs Actual Yield Loss % (R² = {test_r2:.4f})", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Actual Field Yield Loss (%)", fontsize=11)
axes[0].set_ylabel("Random Forest Predicted Loss (%)", fontsize=11)
axes[0].set_xlim(-2, 102)
axes[0].set_ylim(-2, 102)
axes[0].legend(loc="upper left")
axes[0].grid(True, alpha=0.3)

# 2. Residual Distribution
residuals = y_test - y_pred
sns.histplot(residuals, kde=True, ax=axes[1], color="royalblue", bins=25)
axes[1].axvline(0, color="black", linestyle=":", lw=1.5)
axes[1].set_title(f"Residual Error Distribution (MAE = {test_mae:.2f}%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Residual Error (Actual - Predicted %)", fontsize=11)
axes[1].set_ylabel("Frequency", fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 09: Agronomic Feature Importance Analysis
# 1. Gini / MDI (Mean Decrease in Impurity)
mdi_importances = model_rf.feature_importances_

# 2. Permutation Importance (on held-out test set)
perm_res = permutation_importance(model_rf, X_test, y_test, n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1)
perm_mean = perm_res.importances_mean
perm_std = perm_res.importances_std

imp_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "mdi": mdi_importances,
    "perm_mean": perm_mean,
    "perm_std": perm_std
}).sort_values("perm_mean", ascending=True)

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(imp_df))

bars = ax.barh(y_pos, imp_df["perm_mean"], xerr=imp_df["perm_std"], color="#2ca02c", edgecolor="black", alpha=0.85, capsize=4)
ax.set_yticks(y_pos)
ax.set_yticklabels(imp_df["feature"], fontsize=11)
ax.set_xlabel("Permutation Importance (Mean Loss in Test R²)", fontsize=11, fontweight="bold")
ax.set_title("Agronomic Feature Importance Hierarchy in Yield Loss Prediction", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.01, bar.get_y() + bar.get_height()/2.0, f"{w:.3f}", ha="left", va="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Model Persistence & Artifact Packaging
model_export_path = MODELS_DIR / "yield_regressor_rf.joblib"
compat_export_path = MODELS_DIR / "yield_rf_regressor.joblib"

payload = {
    "model": model_rf,
    "features": FEATURE_COLUMNS,
    "metrics": {
        "test_mae": float(test_mae),
        "test_rmse": float(test_rmse),
        "test_r2": float(test_r2)
    },
    "hyperparameters": rf_hyperparameters,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
}

joblib.dump(payload, model_export_path)
joblib.dump(payload, compat_export_path)

print(f"[EXPORT] Model artifact saved to: {model_export_path}")
print(f"[EXPORT] Compatibility artifact saved to: {compat_export_path}")
print(f"[EXPORT] File size: {model_export_path.stat().st_size / (1024*1024):.2f} MB")

# Verify reloadability
loaded_artifact = joblib.load(model_export_path)
assert hasattr(loaded_artifact["model"], "predict"), "Failed to verify loaded model predict capability!"
print("[EXPORT] Artifact reload verified successfully.")

In [ ]:
# Cell 11: Agronomic Loss Advisory Demonstration
sample_cases = [
    {
        "title": "Scenario 1: Early Seedling Infection (Prompt Fungicide Remediation)",
        "features": [12.0, 1, 2.0, 1.2, 0.08, 22.0, 65.0]
    },
    {
        "title": "Scenario 2: Mid-Season Flowering Outbreak (14-Day Remediation Delay)",
        "features": [68.0, 3, 14.0, 2.8, 0.45, 29.0, 85.0]
    },
    {
        "title": "Scenario 3: Late Ripening Blight Outbreak (Near Harvest)",
        "features": [42.0, 6, 7.0, 1.5, 0.30, 24.0, 75.0]
    }
]

print("=================================================================")
print("           CropSure Field Advisory Demonstration                 ")
print("=================================================================")

# Baseline commercial tomato production parameters
AVERAGE_YIELD_TONS_PER_HA = 50.0
MARKET_PRICE_PER_TON_USD = 800.0

for case in sample_cases:
    feat_vector = np.array(case["features"]).reshape(1, -1)
    loss_pred = float(model_rf.predict(feat_vector)[0])
    
    lost_yield_tons = AVERAGE_YIELD_TONS_PER_HA * (loss_pred / 100.0)
    financial_loss_usd = lost_yield_tons * MARKET_PRICE_PER_TON_USD
    
    print(f"\n{case['title']}:")
    print(f"  - Predicted Yield Loss  : {loss_pred:.2f} %")
    print(f"  - Expected Harvest Loss : {lost_yield_tons:.2f} tons / hectare")
    print(f"  - Financial Exposure    : ${financial_loss_usd:,.2f} USD / hectare")
print("=================================================================")

## Provenance Disclosure, Limitations & Agronomic Guidance

### Provenance & Synthetic Nature
This predictive model was trained on **2,000 synthetic observations** mathematically generated from validated epidemiological formulas (AUDPC response kinetics, non-linear phenology vulnerability, and VPD interactions). While it faithfully captures empirical principles known in plant pathology, **it has not been directly calibrated against specific regional micro-trials**.

### Key Limitations
1. **Cultivar Tolerance Variation**: Different tomato hybrids (*indeterminate* greenhouse cultivars vs *determinate* field cultivars) exhibit distinct genetic resistance levels and yield compensation mechanisms not modeled here.
2. **Soil & Nutrient Confounders**: Severe potassium or calcium deficiency can induce physiological symptoms (blossom end rot) that exacerbate apparent disease loss.

### Recalibration Protocol
Before deployment in regional crop insurance underwriting, this pipeline should ingest actual historical harvest yield logs from agricultural research stations and recalibrate the Random Forest weights via transfer learning.